In [ ]:
import example_project as ep

ep.__version__

In [ ]:
import numpy as np
import pandas as pd
import arviz as az

pd.options.plotting.backend = "plotly"

from summer4 import (
    Epoch,
    FlowMass,
    FlowModel,
    Param,
    Property,
    PropertyMap,
    Split,
    Target,
    TargetSet,
    TransitionFlow,
)
from summer4.epi import BayesianModel, ForceOfInfection, Poisson, Uniform

In [ ]:
disease_state = Property("disease_state", ["S", "I", "R"])
humans = PropertyMap.from_property(disease_state)
humans.labels()

In [ ]:
age = Property("age", ["child", "adult"])
age_strat = humans.stratify(age)
age_strat.labels()

In [ ]:
# The force of infection is computed per age group. With no mixing matrix, each age group
# mixes only with itself; pass mixing=MixingMatrix(age, ...) for anything else. Here every age
# group starts with the same share infectious, so this is the same epidemic as homogeneous mixing.
foi = ForceOfInfection(
    "foi", infectious=disease_state["I"], group_by=age, contact_rate=Param("contact_rate")
)

In [ ]:
# Rates are expressions: arithmetic on Params builds the rate, and values arrive at run time.
infection = TransitionFlow("infection", disease_state["S"], disease_state["I"], foi)
recovery = TransitionFlow(
    "recovery", disease_state["I"], disease_state["R"], 1.0 / Param("recovery_time")
)

In [ ]:
epi_model = FlowModel(age_strat)

epi_model.add_flow(infection)
epi_model.add_flow(recovery)

In [ ]:
base_pops = {age["child"]: 1000.0, age["adult"]: 1500.0}
pop_splits = [Split(disease_state, {"S": 0.9, "I": 0.1, "R": 0.0})]

epi_model.set_initial_population(base_pops, pop_splits)

In [ ]:
# An Epoch maps model time (days) to calendar dates, so results come back date-indexed.
times = pd.date_range("7 jun 1980", "7 december 1980")
run_kwargs = dict(t0=0.0, t1=len(times) - 1, dt=1.0, epoch=Epoch(times[0]))

model = epi_model.compile()

In [ ]:
params = {"contact_rate": 0.2, "recovery_time": 20.0}
results = model.run(params, **run_kwargs)

In [ ]:
# Daily infections (summed over age), observed once a week, with a little noise.
inf_target = results["infection"].total().to_pandas()["value"][7:60:7]
inf_target_fuzzy = inf_target * np.exp(np.random.normal(scale=0.01, size=len(inf_target)))
inf_target_fuzzy.plot()

In [ ]:
priors = [
    Uniform("contact_rate", 0.001, 0.5),
    Uniform("recovery_time", 10.0, 50.0),
]

In [ ]:
# The target says what to compare (the infection flow, summed over age) and how to score it.
target = Target.from_series(
    "infection",
    inf_target_fuzzy,
    run_kwargs["epoch"],
    quantity=FlowMass("infection"),
    reduce="sum",
    likelihood=Poisson("sum"),
)
bm = BayesianModel(model, {}, priors, TargetSet((target,)), run_kwargs=run_kwargs)

In [ ]:
mcmc = bm.sample(num_warmup=200, num_samples=200, num_chains=4)

In [ ]:
idata = mcmc.idata

In [ ]:
az.summary(idata)

In [ ]:
az.plot_dist(idata)

In [ ]:
az.plot_trace_dist(idata)